# 单模态基线实验结果

**负责人**：孙钰淼 | **周次**：W14 | **数据**：Florida（预处理后）

## 实验设置
- 数据划分：80% Train / 10% Val / 10% Test（seed=42）
- 结构化特征：882 维（排除 lastSoldPrice 和 listPrice）
- 文本特征：TF-IDF SVD 128维 + BERT CLS 768维
- 硬件：CPU (PyTorch 2.12)


## 实验结果汇总（测试集）

| 模型 | 模态 | RMSE | MAE | R² | MAPE(%) | 时间 |
|------|------|------|------|-----|---------|------|
| **EarlyFusionMLP** | fusion_early | 107,725 | 71,499 | **0.8447** | 228.7 | 29.7s |
| **MidFusionModel** | fusion_mid | 118,124 | 78,640 | **0.8133** | 322.4 | 57.3s |
| EarlyFusionXGBoost | fusion_early | 128,782 | 89,026 | 0.7780 | 195.9 | 9.4s |
| LateFusionStacking | fusion_late | 137,795 | 95,169 | 0.7459 | 336.6 | 18.3s |
| LinearBaseline | structured | 138,424 | 98,184 | 0.7436 | 404.5 | 0.7s |
| XGBoostBaseline | structured | 142,935 | 99,872 | 0.7266 | 373.8 | 1.9s |
| RandomForestBaseline | structured | 164,140 | 110,771 | 0.6394 | 333.8 | 5.6s |
| TFIDFRidgeBaseline | text | 175,621 | 132,936 | 0.5872 | 501.8 | 0.0s |
| BERTMLPBaseline | text | 207,417 | 153,540 | 0.4242 | 709.9 | 41.4s |


In [ ]:
# 可视化：模型对比柱状图
import matplotlib.pyplot as plt
import numpy as np

models = ["Linear", "RF", "XGB", "TFIDF+Ridge", "BERT+MLP",
          "EarlyFusion
XGB", "EarlyFusion
MLP", "MidFusion", "LateFusion"]
r2_scores = [0.7436, 0.6394, 0.7266, 0.5872, 0.4242,
             0.7780, 0.8447, 0.8133, 0.7459]
colors = ["#3498db"]*3 + ["#e74c3c"]*2 + ["#2ecc71"]*2 + ["#9b59b6"] + ["#f39c12"]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(models, r2_scores, color=colors, edgecolor="white")
ax.set_ylabel("R² Score")
ax.set_title("Model Comparison — Test Set R²")
ax.axhline(y=0.7436, color="#3498db", linestyle="--", alpha=0.5, label="Best structured baseline")
ax.legend()
for bar, score in zip(bars, r2_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{score:.4f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()


## 关键发现

1. **融合 > 单模态**：最佳融合模型（EarlyFusionMLP, R²=0.8447）优于最佳单模态（Linear, R²=0.7436），提升 10.1 个百分点
2. **早期融合最优**：直接拼接 + 强大学习器是最有效策略
3. **注意力融合有竞争力**：MidFusionModel 排第2（R²=0.8133）
4. **文本单独预测弱但融合价值大**：TF-IDF 单独 R²=0.59，但与结构化拼接后提升至 0.84
5. **BERT 过拟合**：BERT+MLP 训练 R²=0.75 → 测试 R²=0.42，需更多正则化
6. **LateFusion 效果最差**：Stacking 几乎无增量（+0.23%），基模型预测高度相关


## 结论

- 多模态融合对房价预测任务有显著增量价值
- 早期融合（拼接）+ MLP 是最佳工程选择
- 后续需要超参数调优进一步优化各模型